# S02E02 — Electricity

**Zadanie:** Rozwiązać puzzle elektryczne na planszy 3×3. Doprowadzić prąd do trzech elektrowni  
przez obroty pól (każdy obrót = POST do API). Cel na obrazku: `solved_electricity.png`.

**Techniki:** vision model, interpretacja obrazu, agent z function calling

**Uwaga:** `google/gemini-flash` lub `claude-opus-4-6` radzą sobie najlepiej z tym obrazem.

In [ ]:
import os, base64, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
BOARD_URL  = f"https://hub.ag3nts.org/data/{API_KEY}/electricity.png"
TARGET_URL = "https://hub.ag3nts.org/i/solved_electricity.png"

print("Konfiguracja OK.")

In [ ]:
def fetch_board(reset: bool = False) -> bytes:
    """Pobierz aktualny obraz planszy (opcjonalnie zresetuj)."""
    url = BOARD_URL + ("?reset=1" if reset else "")
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.content


def img_to_b64(img_bytes: bytes) -> str:
    return base64.b64encode(img_bytes).decode()


def rotate_cell(cell: str) -> dict:
    """Wyślij obrót pola (np. '2x3'). Jeden POST = jeden obrót."""
    resp = requests.post(VERIFY_URL, json={
        "apikey": API_KEY,
        "task": "electricity",
        "answer": {"rotate": cell}
    })
    return resp.json()


print("Funkcje gotowe.")

In [ ]:
# Pobierz stan aktualny i docelowy
current_bytes = fetch_board()
target_bytes  = requests.get(TARGET_URL).content

print(f"Aktualna plansza: {len(current_bytes)} bajtów")
print(f"Docelowa plansza: {len(target_bytes)} bajtów")

# Wyświetl (opcjonalnie)
from IPython.display import Image, display
print("Stan aktualny:")
display(Image(data=current_bytes))
print("Stan docelowy:")
display(Image(data=target_bytes))

In [ ]:
# Użyj vision model do opisania obu stanów planszy
analysis_prompt = """Masz dwa obrazki planszy 3x3 z kablami elektrycznymi.
Każde pole adresowane jest jako RxC (wiersz x kolumna, np. 1x1 = lewy górny róg).

Dla KAŻDEGO z 9 pól (1x1, 1x2, 1x3, 2x1, 2x2, 2x3, 3x1, 3x2, 3x3) opisz:
- W obrazku AKTUALNYM: w jakich kierunkach (góra/dół/lewo/prawo) wychodzą kable?
- W obrazku DOCELOWYM: w jakich kierunkach wychodzą kable?
- Ile obrotów o 90° w prawo trzeba wykonać żeby przejść z aktualnego do docelowego?

Odpowiedz w formacie JSON:
{"rotations": {"1x1": 0, "1x2": 1, "1x3": 2, ...}}
Liczba = ile razy obrócić to pole (0 = bez zmian, 1 = 90° w prawo, 2 = 180°, 3 = 270°)."""

resp = client.messages.create(
    model="claude-opus-4-6",
    max_tokens=1000,
    messages=[{
        "role": "user",
        "content": [
            {"type": "text",  "text": "OBRAZ AKTUALNY:"},
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_to_b64(current_bytes)}},
            {"type": "text",  "text": "OBRAZ DOCELOWY:"},
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": img_to_b64(target_bytes)}},
            {"type": "text",  "text": analysis_prompt}
        ]
    }]
)

import json, re
analysis_text = resp.content[0].text
print("Analiza modelu:")
print(analysis_text)

In [ ]:
# Wyciągnij plan obrotów z odpowiedzi
try:
    start = analysis_text.find("{")
    end   = analysis_text.rfind("}") + 1
    plan  = json.loads(analysis_text[start:end])
    rotations = plan.get("rotations", {})
except Exception as e:
    print(f"Błąd parsowania: {e}")
    # Fallback: ręczne wpisanie
    rotations = {"1x1": 0, "1x2": 0, "1x3": 0, "2x1": 0, "2x2": 0, "2x3": 0, "3x1": 0, "3x2": 0, "3x3": 0}

print("Plan obrotów:")
for cell, n in rotations.items():
    if n > 0:
        print(f"  {cell}: {n} obrót/y")

In [ ]:
# Wykonaj obroty
total_rotations = sum(n for n in rotations.values() if n > 0)
print(f"Łączna liczba obrotów do wykonania: {total_rotations}")

flag_found = False
done_count = 0

for cell, n in rotations.items():
    for i in range(n):
        result = rotate_cell(cell)
        done_count += 1
        code = result.get("code", -1)
        print(f"  [{done_count}/{total_rotations}] rotate {cell} -> code={code}")

        if "FLG:" in str(result):
            flags = re.findall(r'\{FLG:[^}]+\}', str(result))
            print(f"\nFLAGA: {flags}")
            flag_found = True
            break

    if flag_found:
        break

if not flag_found:
    print("\nNie znaleziono flagi. Pobierz aktualny stan i sprawdź czy plansza jest poprawna.")
    current_bytes = fetch_board()
    display(Image(data=current_bytes))